# QLoRA Fine-tuning: Qwen2.5-1.5B-Instruct — Retrieval-Augmented, Abstention-Aware QA

**Target:** Kaggle notebook · GPU T4 x2 · Internet enabled · Unattended via *Save & Run All*

---

## ⚠️ Required Manual Steps Before Running

1. Accept dataset access at **https://huggingface.co/datasets/netop/TeleQnA** (one-time, in browser).
2. Add your HF token as a Kaggle Secret named **`HF_TOKEN`** (Notebook → Add-ons → Secrets).

---

## Design Decisions

| Decision | Rationale |
|---|---|
| **No choices in prompt** | Forces extraction from evidence, not copying from a visible option list. |
| **Gold passage guaranteed in positive context** | Previous pipeline excluded it, making extraction unlearnable (Token-F1 ~0.14). |
| **Split before corpus build** | Prevents eval explanations leaking into training distractors via shared TF-IDF corpus. |
| **Answer-contamination filter on ALL negatives** | Any mention of the answer term in a negative passage risks teaching 'evidence contains answer → abstain'. |
| **Distractor false-negative filter** | Skip distractors that contain the answer text. |
| **Mixed hard/easy negatives (50/50)** | Hard negatives (top-similar, answer-free) teach genuine relevance reasoning. |
| **Bare-digit check uses num_choices range** | Legitimate answers can be numbers (e.g. '16'). Only flag digits within valid index range (1..num_choices). |
| **Save BEFORE for_inference()** | Unsloth patches the model for inference (fused kernels). Saving after may capture patched weights. |
| **trl version parsed as (major, minor) tuple** | Plain minor-version int check breaks for trl ≥ 1.0 (minor becomes 0). |
| **Counter-based Token-F1** | Set intersection collapses repeated tokens; Counter multiset intersection is correct. |
| **TF-IDF as placeholder** | Stand-in for 3GPP dense-retrieval pipeline. |
| **No Brier score / early exit** | Out of scope for this SFT phase. |

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Cell 0  —  Install dependencies
# Unsloth manages bitsandbytes / peft / trl / transformers internally.
# Do NOT pin those separately — version conflicts will break the build.
# ═══════════════════════════════════════════════════════════════════════════════
import subprocess, sys
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "unsloth", "rank_bm25", "scikit-learn"],
    check=True,
)
print("Installation complete.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Cell 1  —  GPU check + stage tracker
# ═══════════════════════════════════════════════════════════════════════════════
import torch

assert torch.cuda.is_available(), (
    "CUDA not available. Select 'GPU T4 x2' in Kaggle Settings → Accelerator."
)
print(f"GPU  : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

stage_results = {
    "Environment & Auth" : False,
    "Data Load"          : False,
    "Data Construction"  : False,
    "Model Load"         : False,
    "Training"           : False,
    "Saving"             : False,
    "Evaluation"         : False,
}

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Cell 2  —  Authenticate with Hugging Face via Kaggle Secret
# ═══════════════════════════════════════════════════════════════════════════════
import os
from huggingface_hub import login

try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as _exc:
    raise RuntimeError(
        "Could not read Kaggle Secret 'HF_TOKEN'. "
        "Go to Notebook → Add-ons → Secrets and add your HF access token."
    ) from _exc

if not hf_token or not hf_token.strip():
    raise RuntimeError("Kaggle Secret 'HF_TOKEN' is empty.")

login(token=hf_token)
print("Hugging Face authentication successful.")
stage_results["Environment & Auth"] = True

## Stage 2 — Load and Validate TeleQnA

**Schema notes:**
- Only the `test` split exists — there is **no** `train` split.
- The column is spelled **`explaination`** (source-data typo) — exact spelling used here.
- `answer` is a **1-indexed integer** into `choices`. Resolved as `choices[answer - 1]`.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Cell 3  —  Load dataset and validate schema
# ═══════════════════════════════════════════════════════════════════════════════
from datasets import load_dataset

print("Loading 'netop/TeleQnA' (split='test') …")
try:
    raw_dataset = load_dataset("netop/TeleQnA", split="test")
except Exception as _exc:
    raise RuntimeError(
        "Failed to load dataset. Have you (1) accepted gated access at "
        "https://huggingface.co/datasets/netop/TeleQnA and (2) set HF_TOKEN?"
    ) from _exc

REQUIRED_COLUMNS = {"question", "choices", "answer", "subject", "explaination"}
actual_columns   = set(raw_dataset.column_names)
missing          = REQUIRED_COLUMNS - actual_columns
if missing:
    raise RuntimeError(
        f"Dataset is missing expected columns: {missing}.\n"
        f"Actual columns found: {actual_columns}"
    )

print(f"Loaded {len(raw_dataset):,} records. Columns: {raw_dataset.column_names}")
stage_results["Data Load"] = True

## Stage 3 — Data Construction

### Split Before Corpus Build
90/10 question split is done **before** any TF-IDF index is fitted.
Two independent indices: `train_vec` / `eval_vec`. Eval explanations never appear in training distractors.

### Prompt Format (no choices)
```
### Context:
[1] <passage>
[2] <passage>

### Question:
<question text>

### Answer:
```

### Negative Construction
- 50% hard (top-similar) + 50% easy (bottom-similar).
- **Answer-contamination filter on ALL negatives** (no `use_hard` guard).
- Target: `"I cannot answer this from the provided context."`

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Cell 4  —  Split raw data first; build per-split TF-IDF indices
# ═══════════════════════════════════════════════════════════════════════════════
import random
import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

random.seed(42)
np.random.seed(42)

# ── Global constants ──────────────────────────────────────────────────────────
TARGET_NEGATIVE_RATIO = 0.30
HARD_NEG_RATIO        = 0.50
ABSTAIN_STRING        = "I cannot answer this from the provided context."
MAX_SEQ_LENGTH        = 2048
MODEL_ID              = "Qwen/Qwen2.5-1.5B-Instruct"

# ── Step 1: Question-level split BEFORE any retrieval index ──────────────────
all_raw_indices = list(range(len(raw_dataset)))
train_raw_idx, eval_raw_idx = train_test_split(
    all_raw_indices, test_size=0.10, random_state=42
)
train_raw = raw_dataset.select(train_raw_idx)
eval_raw  = raw_dataset.select(eval_raw_idx)
print(f"Raw split: {len(train_raw):,} train | {len(eval_raw):,} eval questions")

# ── Step 2: Build independent corpora ────────────────────────────────────────
def build_corpus(split):
    """Deduplicated list of non-empty explanation strings."""
    return list(dict.fromkeys(
        p for p in split["explaination"]
        if p and isinstance(p, str) and p.strip()
    ))

train_corpus = build_corpus(train_raw)
eval_corpus  = build_corpus(eval_raw)
assert len(train_corpus) > 0, "Train corpus is empty."
assert len(eval_corpus)  > 0, "Eval corpus is empty."
print(f"Corpus sizes — train: {len(train_corpus):,} | eval: {len(eval_corpus):,} passages")

# ── Step 3: Fit independent TF-IDF indices ───────────────────────────────────
def fit_tfidf_index(corpus, questions):
    vec = TfidfVectorizer(stop_words="english", min_df=1)
    mat = vec.fit_transform(corpus)
    sim = cosine_similarity(vec.transform(questions), mat)
    return vec, mat, sim

print("Building train retrieval index …")
train_vec, train_tfidf, train_sim = fit_tfidf_index(train_corpus, train_raw["question"])
print("Building eval retrieval index …")
eval_vec,  eval_tfidf,  eval_sim  = fit_tfidf_index(eval_corpus,  eval_raw["question"])
print("Retrieval indices ready.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Cell 5  —  Build formatted examples
# ═══════════════════════════════════════════════════════════════════════════════

def resolve_answer(choices, answer_idx: int) -> str:
    """1-indexed integer → answer text string. Never returns a raw digit."""
    return choices[int(answer_idx) - 1]


def build_user_content(context_passages: list, question: str) -> str:
    """
    Prompt template — identical at train and eval time.
    Choices deliberately omitted: model must extract from retrieved passages.
    """
    ctx = "\n".join(f"[{i+1}] {p}" for i, p in enumerate(context_passages))
    return (
        f"### Context:\n{ctx}\n\n"
        f"### Question:\n{question}\n\n"
        f"### Answer:"
    )


def build_examples_for_split(split_raw, sim_matrix, corpus, split_name):
    """
    Returns (examples, labels).

    All dict values are scalars (str / bool / int) — no variable-length lists.
    This avoids Apache Arrow schema inference errors in HF Dataset.from_list().

    num_choices is stored so the sanity check can distinguish a legitimate
    numeric answer (e.g. '16') from an unresolved choice index (e.g. '3'
    when len(choices)==4).
    """
    examples = []
    labels   = []

    for i, record in enumerate(tqdm(split_raw, desc=f"Building {split_name}")):
        question     = record["question"]
        choices      = record["choices"]       # list[str] from dataset
        answer_idx   = int(record["answer"])   # 1-indexed
        gold_passage = record["explaination"]

        if not gold_passage or not isinstance(gold_passage, str) or not gold_passage.strip():
            continue  # skip records with missing explanation

        target_text = resolve_answer(choices, answer_idx)
        is_positive = (random.random() >= TARGET_NEGATIVE_RATIO)

        sim_scores  = sim_matrix[i]
        sorted_idxs = np.argsort(sim_scores)[::-1]   # highest → lowest similarity

        if is_positive:
            # ── POSITIVE: gold passage MUST appear in the context ─────────────
            num_distractors = random.randint(2, 4)
            distractors     = []
            for idx in sorted_idxs:
                p = corpus[idx]
                if p.strip() == gold_passage.strip():
                    continue  # don't duplicate the gold passage
                if target_text.lower() in p.lower():
                    continue  # distractor contains the answer → false-negative, skip
                distractors.append(p)
                if len(distractors) == num_distractors:
                    break

            context_passages = [gold_passage] + distractors
            random.shuffle(context_passages)  # randomise gold position
            target = target_text
            labels.append(1)

        else:
            # ── NEGATIVE: ALL context passages must be answer-free ────────────
            # Filter applies to BOTH hard and easy negatives.
            # Even an incidental mention of the answer term corrupts the signal:
            # the model would see "evidence mentions answer → abstain".
            num_passages   = random.randint(3, 5)
            use_hard       = (random.random() < HARD_NEG_RATIO)
            candidate_idxs = sorted_idxs if use_hard else sorted_idxs[::-1]

            unrelated = []
            for idx in candidate_idxs:
                p = corpus[idx]
                if p.strip() == gold_passage.strip():
                    continue
                if target_text.lower() in p.lower():   # <— NO use_hard guard
                    continue
                unrelated.append(p)
                if len(unrelated) == num_passages:
                    break

            context_passages = unrelated
            random.shuffle(context_passages)
            target = ABSTAIN_STRING
            labels.append(0)

        examples.append({
            "question"    : question,
            "user_content": build_user_content(context_passages, question),
            "target"      : target,
            "gold_passage": gold_passage,
            "is_positive" : is_positive,
            "num_choices" : len(choices),   # needed by sanity check only
        })

    assert len(examples) == len(labels)
    return examples, labels


train_examples, train_labels = build_examples_for_split(
    train_raw, train_sim, train_corpus, "train"
)
eval_examples, eval_labels = build_examples_for_split(
    eval_raw, eval_sim, eval_corpus, "eval"
)

print(f"\nTrain: {len(train_examples):,} examples "
      f"({sum(train_labels):,} pos, {train_labels.count(0):,} neg)")
print(f"Eval : {len(eval_examples):,}  examples "
      f"({sum(eval_labels):,} pos, {eval_labels.count(0):,} neg)")

## Stage 3b — Mandatory Sanity Checks

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Cell 6  —  Mandatory sanity checks
# ═══════════════════════════════════════════════════════════════════════════════
print("Running mandatory sanity checks …\n")


def run_sanity_checks(examples, labels, split_name):

    # ── Check 1: Negative ratio ───────────────────────────────────────────────
    neg_ratio = labels.count(0) / len(labels)
    assert abs(neg_ratio - TARGET_NEGATIVE_RATIO) <= 0.05, (
        f"[{split_name}] Neg ratio {neg_ratio:.3f} deviates >5pp from "
        f"target {TARGET_NEGATIVE_RATIO}"
    )
    print(f"  [{split_name}] Check 1 — Negative ratio: {neg_ratio:.3f} ✓")

    # ── Check 2: Empty targets + unresolved-index guard ───────────────────────
    # WHY NOT plain isdigit(): legitimate answers can be numbers ('16', '256').
    # An unresolved index is a digit whose value falls in [1, num_choices].
    # TeleQnA has ≤6 choices so '16' is always safe; '3' with 4 choices is not.
    for idx, ex in enumerate(examples):
        assert ex["target"].strip() != "", (
            f"[{split_name}] Empty target at index {idx}"
        )
        if ex["is_positive"]:
            t = ex["target"].strip()
            if t.isdigit():
                t_int = int(t)
                assert t_int > ex["num_choices"], (
                    f"[{split_name}] Positive target '{t}' at index {idx} is a digit "
                    f"within valid choice-index range (1..{ex['num_choices']}). "
                    "choices[answer-1] was likely not resolved — check resolve_answer()."
                )
    print(f"  [{split_name}] Check 2 — No empty / unresolved-index targets ✓")

    # ── Check 3: Gold passage in positive context — 100% guaranteed ──────────
    # Fingerprint = first 150 chars of gold passage (guards against very long passages).
    pos_exs     = [ex for ex in examples if ex["is_positive"]]
    sample_size = min(200, len(pos_exs))
    sample      = random.sample(pos_exs, sample_size)

    hits   = 0
    misses = []
    for ex in sample:
        fp = ex["gold_passage"].lower().strip()[:150]
        if fp and fp in ex["user_content"].lower():
            hits += 1
        else:
            misses.append(ex["question"][:80])

    rate = hits / sample_size
    print(f"  [{split_name}] Check 3 — Gold passage in context: {rate:.1%} (n={sample_size})")
    if rate < 1.0:
        print(f"    ↳ Failing questions (first 3): {misses[:3]}")
    assert rate == 1.0, (
        f"[{split_name}] Gold passage in only {rate:.1%} of prompts — "
        "must be 100%. Check build_examples_for_split()."
    )

    # ── Check 4: All negative targets == abstain string ───────────────────────
    neg_exs = [ex for ex in examples if not ex["is_positive"]]
    bad     = [ex for ex in neg_exs if ex["target"] != ABSTAIN_STRING]
    assert len(bad) == 0, (
        f"[{split_name}] {len(bad)} negative examples have non-abstain targets!"
    )
    print(f"  [{split_name}] Check 4 — All negative targets are abstain string ✓\n")


run_sanity_checks(train_examples, train_labels, "TRAIN")
run_sanity_checks(eval_examples,  eval_labels,  "EVAL")
print("All sanity checks passed. ✓")
stage_results["Data Construction"] = True

## Stage 3c — Diagnostic: Gold Passage Answer Support

Guaranteeing gold passage presence ≠ guaranteeing it contains the answer text.  
TeleQnA explanations sometimes paraphrase rather than verbatim-state the answer.

**This cell is diagnostic only.** It prints three levels of support and samples paraphrase cases.  
String absence ≠ non-support (paraphrase is valid reasoning signal).  
Set `FILTER_UNSUPPORTED_POSITIVES = True` only after reviewing the sample cases below.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Cell 6b  —  Diagnostic: gold-passage answer-support analysis
# ═══════════════════════════════════════════════════════════════════════════════

FILTER_UNSUPPORTED_POSITIVES = False  # review sample output BEFORE enabling


def normalize_text(s: str) -> str:
    """Lowercase, collapse punctuation, strip extra whitespace."""
    s = s.lower()
    s = re.sub(r"[^\w\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def analyse_support(examples, split_name):
    pos_exs    = [ex for ex in examples if ex["is_positive"]]
    exact_hits = 0
    norm_hits  = 0
    neither    = []

    for ex in pos_exs:
        t  = ex["target"]
        gp = ex["gold_passage"]
        if t.lower().strip() in gp.lower():
            exact_hits += 1
        elif normalize_text(t) in normalize_text(gp):
            norm_hits += 1
        else:
            neither.append(ex)

    total = len(pos_exs)
    n_cnt = len(neither)
    print(f"\n[{split_name}] Gold-passage answer support (n={total:,} positive examples):")
    print(f"  Exact verbatim match   : {exact_hits:>5,}  ({exact_hits/total:.1%})")
    print(f"  Normalized match only  : {norm_hits:>5,}  ({norm_hits/total:.1%})")
    print(f"  Neither (paraphrase?)  : {n_cnt:>5,}  ({n_cnt/total:.1%})")

    if neither:
        print(f"\n  Sample 'neither' cases (first 3 of {n_cnt}):")
        for ex in neither[:3]:
            print(f"    Q      : {ex['question'][:90]}")
            print(f"    Target : {ex['target']}")
            print(f"    Passage: {ex['gold_passage'][:160]} …")
            print()
    return neither


train_neither = analyse_support(train_examples, "TRAIN")
eval_neither  = analyse_support(eval_examples,  "EVAL")

if FILTER_UNSUPPORTED_POSITIVES:
    bad_q_train = {ex["question"] for ex in train_neither}
    bad_q_eval  = {ex["question"] for ex in eval_neither}
    before_t = len(train_examples)
    before_e = len(eval_examples)
    train_examples = [ex for ex in train_examples
                      if not ex["is_positive"] or ex["question"] not in bad_q_train]
    eval_examples  = [ex for ex in eval_examples
                      if not ex["is_positive"] or ex["question"] not in bad_q_eval]
    train_labels   = [1 if ex["is_positive"] else 0 for ex in train_examples]
    eval_labels    = [1 if ex["is_positive"] else 0 for ex in eval_examples]
    print(f"\nFiltered train: {before_t:,} → {len(train_examples):,} examples.")
    print(f"Filtered eval : {before_e:,} → {len(eval_examples):,}  examples.")
    for s_name, lbls in [("TRAIN", train_labels), ("EVAL", eval_labels)]:
        nr = lbls.count(0) / len(lbls)
        assert abs(nr - TARGET_NEGATIVE_RATIO) <= 0.08, (
            f"[{s_name}] Neg ratio {nr:.3f} drifted too far after filtering."
        )
        print(f"  [{s_name}] Neg ratio after filter: {nr:.3f} ✓")
else:
    print("\nFILTER_UNSUPPORTED_POSITIVES=False — all positives retained.")
    print("Review the 'neither' sample cases above before enabling the filter.")

## Stage 4 — Load Model and Apply LoRA Adapters

**Model:** `Qwen/Qwen2.5-1.5B-Instruct` · 4-bit NF4 QLoRA via Unsloth  
**LoRA:** r=16, alpha=16, dropout=0, all attention + MLP projections, Unsloth gradient checkpointing

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Cell 7  —  Load base model and attach LoRA adapters
# ═══════════════════════════════════════════════════════════════════════════════
from unsloth import FastLanguageModel

print(f"Loading {MODEL_ID} (4-bit NF4 QLoRA) via Unsloth …")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_ID,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit   = True,
    dtype          = None,
)

print("Attaching LoRA adapters …")
model = FastLanguageModel.get_peft_model(
    model,
    r                          = 16,
    lora_alpha                 = 16,
    lora_dropout               = 0,
    target_modules             = ["q_proj", "k_proj", "v_proj", "o_proj",
                                   "gate_proj", "up_proj", "down_proj"],
    bias                       = "none",
    use_gradient_checkpointing = "unsloth",
    random_state               = 42,
    use_rslora                 = False,
    loftq_config               = None,
)

model.print_trainable_parameters()
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
assert trainable > 0, "No trainable parameters — LoRA attachment failed."
print(f"Trainable parameters: {trainable:,} ✓")
stage_results["Model Load"] = True

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Cell 8  —  Apply chat template and build HF Datasets
# All columns passed to Dataset.from_list() are scalars (str / bool / int).
# Variable-length lists (e.g. choices) are NOT stored — they cause Arrow errors.
# num_choices is dropped here because it was only needed for the sanity check.
# ═══════════════════════════════════════════════════════════════════════════════
import datasets as hf_datasets


def apply_template(example):
    messages = [
        {"role": "user",      "content": example["user_content"]},
        {"role": "assistant", "content": example["target"]},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    return {
        "text"        : text,
        "question"    : example["question"],
        "user_content": example["user_content"],
        "target"      : example["target"],
        "gold_passage": example["gold_passage"],
        "is_positive" : example["is_positive"],
        # num_choices intentionally excluded — not needed after sanity checks
    }


print("Building HF Datasets and applying chat template …")
train_ds = hf_datasets.Dataset.from_list(train_examples).map(
    apply_template, desc="Train template"
)
eval_ds = hf_datasets.Dataset.from_list(eval_examples).map(
    apply_template, desc="Eval template"
)

train_neg = 1.0 - (sum(train_ds["is_positive"]) / len(train_ds))
eval_neg  = 1.0 - (sum(eval_ds["is_positive"])  / len(eval_ds))
assert abs(train_neg - TARGET_NEGATIVE_RATIO) <= 0.08, (
    f"Train neg ratio drifted: {train_neg:.3f}"
)
assert abs(eval_neg - TARGET_NEGATIVE_RATIO) <= 0.08, (
    f"Eval neg ratio drifted: {eval_neg:.3f}"
)
print(f"Train: {len(train_ds):,} examples | neg ratio: {train_neg:.3f} ✓")
print(f"Eval : {len(eval_ds):,}  examples | neg ratio: {eval_neg:.3f} ✓")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Cell 9  —  Truncation analysis
# ═══════════════════════════════════════════════════════════════════════════════
print(f"Analysing token lengths (MAX_SEQ_LENGTH={MAX_SEQ_LENGTH}) …")

truncated     = 0
gold_cutoff   = 0
answer_cutoff = 0
all_ds        = hf_datasets.concatenate_datasets([train_ds, eval_ds])

for ex in tqdm(all_ds, desc="Token length check"):
    ids = tokenizer(
        ex["text"], truncation=False, padding=False, add_special_tokens=False
    )["input_ids"]
    if len(ids) > MAX_SEQ_LENGTH:
        truncated += 1
        if ex["is_positive"]:
            surviving = tokenizer.decode(
                ids[:MAX_SEQ_LENGTH], skip_special_tokens=True
            ).lower()
            if ex["gold_passage"].lower().strip()[:150] not in surviving:
                gold_cutoff += 1
            if ex["target"].lower().strip() not in surviving:
                answer_cutoff += 1

total = len(all_ds)
del all_ds  # free memory — not needed again

print(f"  Truncated      : {truncated:,} / {total:,} ({truncated/total:.1%})")
if truncated > 0:
    print(f"  ⚠  WARNING: {truncated/total:.1%} of examples will be truncated.")
    if gold_cutoff   > 0:
        print(f"  🔴 CRITICAL: Gold passage cut in {gold_cutoff:,} positive examples!")
    if answer_cutoff > 0:
        print(f"  🔴 CRITICAL: Answer span cut in {answer_cutoff:,} positive examples!")
else:
    print("  All examples fit within MAX_SEQ_LENGTH. ✓")

## Stage 5 — Training

**trl version handling:** Parsed as `(major, minor)` tuple so `trl ≥ 1.0` is handled correctly.  
Plain minor-integer check (`int(ver.split('.')[1]) >= 9`) breaks for trl 1.x (minor → 0).  

**Memory:** `torch.cuda.empty_cache()` called before training to release data-construction allocations.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Cell 10  —  Configure and launch SFTTrainer
# ═══════════════════════════════════════════════════════════════════════════════
import trl as _trl
from trl import SFTTrainer
from transformers import TrainingArguments

OUTPUT_DIR = "outputs"

# ── Free memory from data construction before loading training buffers ────────
torch.cuda.empty_cache()

# ── Resume from latest checkpoint if one exists ───────────────────────────────
resume_checkpoint = False
if os.path.isdir(OUTPUT_DIR):
    _ckpts = sorted(
        [d for d in os.listdir(OUTPUT_DIR) if d.startswith("checkpoint-")],
        key=lambda x: int(x.split("-")[1]),
    )
    if _ckpts:
        resume_checkpoint = os.path.join(OUTPUT_DIR, _ckpts[-1])
        print(f"Resuming from: {resume_checkpoint}")

# ── trl version: parse as (major, minor) tuple ────────────────────────────────
# Plain int(ver.split('.')[1]) breaks for trl >= 1.0 (minor becomes 0).
# processing_class= was introduced in trl 0.9.0 and kept in all later versions.
_m = re.match(r"(\d+)\.(\d+)", _trl.__version__)
assert _m, f"Could not parse trl version: '{_trl.__version__}'"
_trl_ver  = (int(_m.group(1)), int(_m.group(2)))
tok_kwarg = (
    {"processing_class": tokenizer}
    if _trl_ver >= (0, 9)
    else {"tokenizer": tokenizer}
)
print(f"trl version: {_trl.__version__}  → using key: '{list(tok_kwarg.keys())[0]}'")

trainer = SFTTrainer(
    model              = model,
    **tok_kwarg,
    train_dataset      = train_ds,
    eval_dataset       = eval_ds,
    dataset_text_field = "text",
    max_seq_length     = MAX_SEQ_LENGTH,
    dataset_num_proc   = 2,
    packing            = False,
    args = TrainingArguments(
        output_dir                  = OUTPUT_DIR,
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,
        num_train_epochs            = 3,
        learning_rate               = 2e-4,
        bf16                        = torch.cuda.is_bf16_supported(),
        fp16                        = not torch.cuda.is_bf16_supported(),
        optim                       = "adamw_8bit",
        save_steps                  = 100,
        save_total_limit            = 2,
        logging_steps               = 10,
        report_to                   = "none",
        seed                        = 42,
    ),
)

print("Starting training …")
trainer.train(resume_from_checkpoint=resume_checkpoint)

assert trainer.state.global_step > 0, (
    f"global_step={trainer.state.global_step} — nothing was trained."
)
print(f"Training complete. global_step = {trainer.state.global_step} ✓")
stage_results["Training"] = True

## Stage 6 — Save LoRA Adapter and 16-bit Merged Model

**Critical ordering:** Save happens **before** `FastLanguageModel.for_inference()`.  
Unsloth patches the model for fast inference (fused kernels).  
Calling `save_pretrained_merged` after `for_inference()` may capture patched weights.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Cell 11  —  Save LoRA adapter and 16-bit merged model  (BEFORE for_inference)
# ═══════════════════════════════════════════════════════════════════════════════
LORA_PATH   = "qlora_adapter"
MERGED_PATH = "qwen2.5_teleqna_merged"

print(f"Saving LoRA adapter → '{LORA_PATH}' …")
model.save_pretrained(LORA_PATH)
tokenizer.save_pretrained(LORA_PATH)

print(f"Saving 16-bit merged model → '{MERGED_PATH}' …")
model.save_pretrained_merged(MERGED_PATH, tokenizer, save_method="merged_16bit")


def dir_size_mb(path: str) -> float:
    total = 0
    for dp, _, fnames in os.walk(path):
        for fn in fnames:
            fp = os.path.join(dp, fn)
            if not os.path.islink(fp):
                total += os.path.getsize(fp)
    return total / (1024 * 1024)


assert os.path.isdir(LORA_PATH),   f"LoRA dir '{LORA_PATH}' missing after save!"
assert os.path.isdir(MERGED_PATH), f"Merged dir '{MERGED_PATH}' missing after save!"

lora_mb   = dir_size_mb(LORA_PATH)
merged_mb = dir_size_mb(MERGED_PATH)
print(f"LoRA adapter size  : {lora_mb:.1f} MB")
print(f"Merged model size  : {merged_mb:.1f} MB")

assert lora_mb   > 10,   f"LoRA {lora_mb:.1f} MB — suspiciously small."
assert merged_mb > 2000, (
    f"Merged {merged_mb:.1f} MB — too small for 1.5B 16-bit (~3 GB expected)."
)
print("Both artefacts saved and size-verified. ✓")
stage_results["Saving"] = True

## Stage 7 — Evaluation

**Four metrics:**
1. Positive Exact Match — strict string equality on answerable examples.
2. Positive Token-F1 — `Counter` multiset intersection (NOT set — set collapses repeated tokens).
3. Abstention Recall — of gold-abstain, fraction correctly caught.
4. Abstention Precision — of model-answered, fraction that were actually answerable.

**Safety:** `attention_mask` passed explicitly (Qwen2.5 `pad_token_id == eos_token_id`).  
`for_inference()` called here, AFTER saving is complete.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Cell 12  —  Evaluation  (for_inference called here, AFTER saving)
# ═══════════════════════════════════════════════════════════════════════════════
from collections import Counter

# Switch to fast inference mode AFTER saving is done.
FastLanguageModel.for_inference(model)


def token_f1(pred: str, gold: str) -> float:
    """
    SQuAD-style Token-F1 using Counter multiset intersection.
    DO NOT replace with set intersection — it collapses repeated tokens
    and produces silently wrong (lower) scores.
    """
    pt = pred.lower().split()
    gt = gold.lower().split()
    if not pt or not gt:
        return 0.0
    nc = sum((Counter(pt) & Counter(gt)).values())
    if nc == 0:
        return 0.0
    p = nc / len(pt)
    r = nc / len(gt)
    return (2 * p * r) / (p + r)


print("Starting evaluation …")

exact_matches               = 0
f1_scores                   = []
gold_abstain_total          = 0
gold_abstain_correct        = 0
answered_total              = 0
answered_wrongly_on_abstain = 0
failures                    = 0
wrong_samples               = []

for ex in tqdm(eval_ds, desc="Evaluating"):
    try:
        # apply_chat_template with return_dict=True produces attention_mask.
        # Qwen2.5: pad_token_id == eos_token_id — omitting attention_mask is unreliable.
        inputs = tokenizer.apply_chat_template(
            [{"role": "user", "content": ex["user_content"]}],
            tokenize           = True,
            add_generation_prompt = True,
            return_dict        = True,
            return_tensors     = "pt",
        )
        # Move tensors to device individually (dict has no .to())
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        out_ids = model.generate(
            input_ids      = inputs["input_ids"],
            attention_mask = inputs["attention_mask"],
            max_new_tokens = 128,
            pad_token_id   = tokenizer.eos_token_id,
            do_sample      = False,
        )

        pred_text = tokenizer.decode(
            out_ids[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True,
        ).strip()

        gold_text       = ex["target"].strip()
        is_gold_abstain = not ex["is_positive"]
        is_pred_abstain = (pred_text == ABSTAIN_STRING)

        # Metrics 1 & 2: positive examples only
        if not is_gold_abstain:
            if pred_text == gold_text:
                exact_matches += 1
            f1_scores.append(token_f1(pred_text, gold_text))
            if (pred_text != gold_text
                    and random.random() < 0.05
                    and len(wrong_samples) < 20):
                wrong_samples.append((ex["question"], gold_text, pred_text))

        # Metric 3: abstention recall
        if is_gold_abstain:
            gold_abstain_total += 1
            if is_pred_abstain:
                gold_abstain_correct += 1

        # Metric 4: abstention precision
        if not is_pred_abstain:
            answered_total += 1
            if is_gold_abstain:
                answered_wrongly_on_abstain += 1

    except Exception as _exc:
        print(f"  Generation failure (skipping): {_exc}")
        failures += 1

n_pos      = len(f1_scores)
pos_em     = exact_matches / n_pos if n_pos > 0 else 0.0
pos_f1     = sum(f1_scores) / n_pos if n_pos > 0 else 0.0
abs_recall = (gold_abstain_correct / gold_abstain_total
              if gold_abstain_total > 0 else 0.0)
abs_prec   = ((answered_total - answered_wrongly_on_abstain) / answered_total
              if answered_total > 0 else 0.0)

print("\n" + "=" * 52)
print("             EVALUATION RESULTS")
print("=" * 52)
print(f"  1. Positive Exact Match      : {pos_em:.4f}  ({pos_em:.2%})")
print(f"  2. Positive Token-F1         : {pos_f1:.4f}  ({pos_f1:.2%})")
print(f"  3. Abstention Recall         : {abs_recall:.4f}  ({abs_recall:.2%})")
print(f"  4. Abstention Precision      : {abs_prec:.4f}  ({abs_prec:.2%})")
print(f"     Generation failures        : {failures}")
print("=" * 52)

if wrong_samples:
    print("\nSample wrong positive predictions (≤20, ~5% rate):")
    for q, g, p in wrong_samples:
        print(f"  Q    : {q}")
        print(f"  Gold : {g}")
        print(f"  Pred : {p}")
        print("  " + "-" * 46)

stage_results["Evaluation"] = True

## Final Summary — Check Here First After Overnight Run

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Cell 13  —  Unattended run summary
# ═══════════════════════════════════════════════════════════════════════════════
print("=" * 52)
print("          UNATTENDED RUN SUMMARY")
print("=" * 52)
all_ok = True
for stage, ok in stage_results.items():
    mark   = "✅  PASS" if ok else "❌  FAIL / DID NOT REACH"
    all_ok = all_ok and ok
    print(f"  {stage:<28} {mark}")
print("=" * 52)
if all_ok:
    print("All stages completed successfully.")
    print(f"  LoRA adapter  → {LORA_PATH}/")
    print(f"  Merged model  → {MERGED_PATH}/")
else:
    print("⚠  One or more stages did not complete. Scroll up for error details.")